# Sentinel-2 Cloud-Free Composite Export — Paoay, Atok, Benguet

Builds **half-month (H1/H2) cloud-masked Sentinel-2 median composites** for the study area using Google Earth Engine (GEE), and exports each one to Google Drive as a GeoTIFF. Also includes a coverage check that reports how much of the area survives cloud masking in each period.

**Data sources**
- `COPERNICUS/S2_SR_HARMONIZED` — Sentinel-2 surface reflectance
- `GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED` — Cloud Score+ (per-pixel clear-sky score)

**Contents**
1. Setup — imports and Earth Engine authentication
2. Configuration — area of interest, years, thresholds, export settings
3. Helper functions — cloud masking
4. Map preview — visual sanity check
5. Export composites — main loop → Google Drive
6. Export summary — what was exported / skipped
7. Cloud coverage check — valid-pixel % per period

> **Before running:** see `README.md` for environment setup. Change `GEE_PROJECT` in the configuration cell to your own Earth Engine project.

## 1. Setup

Imports and Earth Engine login. `ee.Authenticate()` opens a browser prompt — it is only strictly needed the first time (or when your token expires), after which `ee.Initialize()` alone is enough.

In [ ]:
import calendar

import ee
import geemap
import pandas as pd
from tqdm import tqdm

from pathlib import Path
import numpy as np
import rasterio

In [ ]:
GEE_PROJECT = "thesis-18"  # <-- replace with your own Earth Engine project ID

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print(ee.String("Earth Engine is working!").getInfo())

## 2. Configuration

All tunable settings live here.

| Setting | Meaning |
|---|---|
| `paoay_aoi` | Study area rectangle `[west, south, east, north]` |
| `years` | Years to process |
| `CS_THRESHOLD` | Cloud Score+ cutoff (0–1). Higher = stricter → fewer but cleaner pixels |
| `BANDS` | Sentinel-2 bands kept in the export |
| `SCALE` | Export resolution in metres |
| `DRIVE_FOLDER` | Google Drive folder that receives the exports |

> **TODO:** replace the rectangle with the actual Paoay shapefile / boundary geometry.

### 2.1 Area of interest

In [ ]:
# Area of Interest — Paoay, Atok, Benguet
# TODO: replace with the actual Paoay boundary (shapefile) instead of a bounding rectangle
paoay_aoi = geemap.shp_to_ee("Paoay_SHP_FILE/Boundary_Paoay_ATOK.shp") 

### 2.2 Processing parameters

In [ ]:
years = [2019, 2020, 2021, 2022, 2023, 2024]

# Cloud Score+ collection (matched to source images by system:index)
cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
CS_THRESHOLD = 0.6  # 0-1, higher = stricter (fewer, cleaner pixels)

# Export settings
BANDS = ["B3", "B4", "B5", "B8", "B11"]
SCALE = 10  # metres
DRIVE_FOLDER = "THS-SENTINEL2"
FILE_PREFIX = "Sentinel2_CSPlus"  # prefix for exported file names

### 2.3 Time periods

Each month is split into two windows: **H1** (1st–16th) and **H2** (16th–end of month), giving 24 periods per year.

In [ ]:
periods_list = []
for year in years:
    for month in range(1, 13):
        last_day = calendar.monthrange(year, month)[1]
        periods_list.append((year, month, f"{year}-{month:02d}-01", f"{year}-{month:02d}-16", "H1"))
        periods_list.append((year, month, f"{year}-{month:02d}-16", f"{year}-{month:02d}-{last_day:02d}", "H2"))

print(f"{len(periods_list)} periods to process")

## 3. Helper functions

`mask_clouds_cs_plus` masks every pixel whose Cloud Score+ (`cs_cdf` band) is below `CS_THRESHOLD`. It is mapped over each image collection after linking with the Cloud Score+ collection.

In [ ]:
def mask_clouds_cs_plus(image):
    """Attach Cloud Score+ and mask pixels below the clear-sky threshold."""
    cs = image.select("cs_cdf")
    return image.updateMask(cs.gte(CS_THRESHOLD))

## 4. Map preview

Creates the interactive map used as a visual sanity check. Composite layers are added to it during the export loop in Section 5 (hidden by default — toggle them from the layer control). The map is displayed at the end of Section 6.

In [ ]:
Map = geemap.Map()
Map.add_basemap("SATELLITE")
Map.add_layer(paoay_aoi, {"color": "red"}, "Study Area")
Map.centerObject(paoay_aoi, 13)

vis_params = {
    "bands": ["B4", "B3", "B5"],
    "min": 0,
    "max": 3000,
}

## 5. Export composites

For every period this cell does the following:

1. **Filter** Sentinel-2 SR scenes by area, date, and scene-level cloud % (< 80 %, drops only near-unusable scenes)
2. **Mask** clouds per pixel with Cloud Score+
3. **Composite** using the per-pixel median of clear pixels, clipped to the AOI
4. **Skip** the period if there is no imagery, or if every pixel was masked
5. **Export** the composite to Google Drive (`DRIVE_FOLDER`) as `Sentinel2_CSPlus_<year>_<month>_<H1|H2>`

> ⏱ This loop calls `.getInfo()` several times per period, so it takes a while. Exports run as background GEE tasks — track them at https://code.earthengine.google.com/tasks.

In [ ]:
summary = []

for year, month, start_date, end_date, half in periods_list:
    label = f"{year}_{month:02d}_{half}"
    print(f"\n{'='*60}\nPROCESSING {label}\n{'='*60}")

    # Raw collection for this window
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(paoay_aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))  # drop only near-unusable scenes
    )

    raw_count = collection.size().getInfo()
    print(f"  Raw scenes found: {raw_count}")

    if raw_count == 0:
        print("  → Skipped: no imagery at all this period")
        summary.append({"label": label, "raw_count": 0, "status": "no_imagery"})
        continue

    # Attach Cloud Score+ and mask
    masked_collection = collection.linkCollection(cs_plus, ["cs_cdf"]).map(mask_clouds_cs_plus)

    # Composite: per-pixel median of clear pixels only
    composite = masked_collection.median().clip(paoay_aoi)
    raw_bands = composite.select(BANDS)

    # Check how much of the AOI actually has valid (non-masked) data
    valid_pixel_pct = raw_bands.select("B4").reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=paoay_aoi,
        scale=SCALE,
        maxPixels=1e13,
    ).get("B4")

    valid_count = ee.Number(valid_pixel_pct).getInfo()
    print(f"  Valid (unmasked) pixel count in B4: {valid_count}")

    if valid_count == 0:
        print("  → Skipped: fully masked, no clear pixels survived")
        summary.append({"label": label, "raw_count": raw_count, "status": "fully_masked"})
        continue

    # Add to map for visual check
    Map.addLayer(raw_bands, vis_params, f"S2 {label}", False)  # False = off by default, toggle manually

    # Export
    task = ee.batch.Export.image.toDrive(
        image=raw_bands,
        description=f"{FILE_PREFIX}_{label}",
        folder=DRIVE_FOLDER,
        fileNamePrefix=f"{FILE_PREFIX}_{label}",
        region=paoay_aoi.geometry(),
        scale=SCALE,
        maxPixels=1e13,
    )
    task.start()
    print(f"  → Export started: {FILE_PREFIX}_{label}")

    summary.append({"label": label, "raw_count": raw_count, "status": "exported"})

## 6. Export summary

Counts how many periods were exported vs. skipped, lists the problem periods, and shows the map.

In [ ]:
print("=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)

exported = [s for s in summary if s["status"] == "exported"]
skipped_empty = [s for s in summary if s["status"] == "no_imagery"]
skipped_masked = [s for s in summary if s["status"] == "fully_masked"]

print(f"Exported: {len(exported)}")
print(f"Skipped (no imagery): {len(skipped_empty)}")
print(f"Skipped (fully masked): {len(skipped_masked)}")

if skipped_empty or skipped_masked:
    print("\nProblem periods:")
    for s in skipped_empty + skipped_masked:
        print(f"  - {s['label']} ({s['status']})")

In [ ]:
display(Map)

## 7. Cloud coverage check

Independent of the export: for each period, computes the **percentage of AOI pixels that remain valid after Cloud Score+ masking**, and saves the result to `cloudscore_coverage_check.csv`.

| Status | Meaning |
|---|---|
| `good` | ≥ 80 % valid pixels |
| `partial` | 0–80 % valid pixels |
| `fully_masked` | no valid pixels |
| `no_imagery` | no scenes found for the period |

This uses a looser scene filter (cloud < 95 %) than the export (< 80 %), so it can flag periods that the export loop would drop.

In [ ]:
results = []

for year, month, start_date, end_date, half in tqdm(periods_list, desc="Checking coverage"):
    label = f"{year}_{month:02d}_{half}"

    # Loose scene-level filter — just to skip obviously hopeless scenes
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(paoay_aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 95))
    )
    raw_count = collection.size().getInfo()

    if raw_count == 0:
        results.append({"label": label, "raw_count": 0, "valid_pixel_pct": 0, "status": "no_imagery"})
        continue

    # Apply real per-pixel cloud masking
    masked_collection = collection.linkCollection(cs_plus, ["cs_cdf"]).map(mask_clouds_cs_plus)
    composite = masked_collection.median().clip(paoay_aoi).select("B4")

    # Count valid (non-masked) pixels vs total pixels in AOI
    valid_count = ee.Number(composite.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=paoay_aoi,
        scale=SCALE,
        maxPixels=1e13,
    ).get("B4")).getInfo()

    total_count = ee.Number(ee.Image(1).clip(paoay_aoi).reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=paoay_aoi,
        scale=SCALE,
        maxPixels=1e13,
    ).get("constant")).getInfo()

    valid_pct = round((valid_count / total_count) * 100, 1) if total_count > 0 else 0

    status = "good" if valid_pct >= 80 else ("partial" if valid_pct > 0 else "fully_masked")

    results.append({
        "label": label,
        "raw_count": raw_count,
        "valid_pixel_pct": valid_pct,
        "status": status,
    })

df = pd.DataFrame(results)
df.to_csv("cloudscore_coverage_check.csv", index=False)

print("\nSaved to cloudscore_coverage_check.csv\n")

### 7.1 Results

In [ ]:
print("Status breakdown:")
print(df["status"].value_counts())

print("\nWorst periods (lowest valid pixel %):")
print(df.sort_values("valid_pixel_pct").head(15)[["label", "raw_count", "valid_pixel_pct", "status"]])

## 8. Spectral indices (NDVI, NDRE, NDMI ) 

In [ ]:
INDEX_DEFS = {
    "NDVI": ("B8", "B4"),   # NIR, Red
    "NDRE": ("B8", "B5"),   # NIR, Red Edge 1
    "NDMI": ("B8", "B11"),  # NIR, SWIR1
}

FEATURE_BANDS = BANDS + list(INDEX_DEFS)

Append NDVI, NDRE and NDMI bands to a composite that has B4, B5, B8, B11.

In [ ]:
# Find the Drive folder automatically (works for any Google account signed in to Drive for Desktop)
matches = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/My Drive/THS-SENTINEL2"))

if len(matches) == 1:
    RAW_DIR = matches[0]
elif not matches:
    raise FileNotFoundError(
        "Could not find 'THS-SENTINEL2' in Google Drive for Desktop. "
        "Is Drive running, and is the folder directly under My Drive?"
    )
else:
    raise RuntimeError(f"Found several matches, set RAW_DIR by hand: {matches}")

print("Using:", RAW_DIR)

OUT_DIR = Path("data/sentinel2_indices")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREFIX = "S2_Indices"

# Reuses BANDS, FILE_PREFIX and periods_list defined in Section 2.
BAND_POS = {name: i for i, name in enumerate(BANDS)}   # 0-based index into the array



In [ ]:
def norm_diff(a, b):
    """Normalized difference, NaN where the denominator is 0."""
    denom = a + b
    out = np.full(a.shape, np.nan, dtype="float32")
    np.divide(a - b, denom, out=out, where=denom != 0)
    return out

In [ ]:
def compute_indices(src_path, out_path):
    """Read a raw composite, write a 3-band NDVI/NDRE/NDMI GeoTIFF on the same grid."""
    with rasterio.open(src_path) as src:
        if src.count != len(BANDS):
            raise ValueError(f"{src_path.name}: expected {len(BANDS)} bands, found {src.count}")
        data = src.read().astype("float32")
        profile = src.profile.copy()

    # Masked pixels come out of the GEE export as 0 in every band (or NaN). Mark them invalid.
    invalid = np.isnan(data).any(axis=0) | (data == 0).all(axis=0)

    layers = []
    for name, (a, b) in INDEX_DEFS.items():
        idx = norm_diff(data[BAND_POS[a]], data[BAND_POS[b]])
        idx[invalid] = np.nan
        layers.append(idx)

    profile.update(count=len(layers), dtype="float32", nodata=np.nan, compress="lzw")
    with rasterio.open(out_path, "w", **profile) as dst:
        for i, (name, layer) in enumerate(zip(INDEX_DEFS, layers), start=1):
            dst.write(layer, i)
            dst.set_band_description(i, name)